# Notebook 01 — Preprocessing

This notebook demonstrates the preprocessing pipeline for the NER OFAC project.

**What this covers:**
- Ingesting mock sentence data (no real Spark/AWS required)
- Language filtering with `langdetect` — keeping only English sentences
- Text normalization using Unicode NFKD decomposition + ASCII encoding
- The full `preprocess()` pipeline combining both steps

**Why preprocessing matters:** Raw data from `ecata.sentences` may contain
accented characters, non-English text, or mixed encodings that confuse
Comprehend's entity detection.  Normalizing text before inference improves
entity boundary detection and match quality.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from ner_ofac.preprocess import normalize_text, filter_english, preprocess

## 1. Mock Data

We create a small pandas DataFrame to simulate the `ecata.sentences` Spark table.
In production this would be a PySpark DataFrame loaded via `ingest.load_sentences()`.

In [ ]:
mock_sentences = pd.DataFrame({
    'a_number': ['A001', 'A002', 'A003', 'A004', 'A005'],
    'source': ['doc1', 'doc1', 'doc2', 'doc3', 'doc3'],
    'line': [1, 2, 1, 1, 2],
    'section': ['intro', 'body', 'intro', 'intro', 'body'],
    'text': [
        'JOHN DOE was added to the sanctions list.',
        'Résumé de la réunion du conseil.',          # French
        'The vessel DARK STAR was seized last week.',
        'Zusammenfassung der Ermittlungsergebnisse.',  # German
        'Müller GmbH was identified as a front company.',
    ]
})

print(f"Total rows: {len(mock_sentences)}")
mock_sentences

## 2. Text Normalization

`normalize_text()` applies Unicode NFKD decomposition followed by ASCII encoding.
This strips diacritics (accents) while preserving case and punctuation.
This is important because entity names in the OFAC list are ASCII-only.

In [ ]:
test_cases = [
    'Résumé de la réunion',
    'Müller GmbH',
    'Ångström Measurement',
    'JOHN DOE was sanctioned.',
    '',
    '中文测试',
]

for text in test_cases:
    normalized = normalize_text(text)
    print(f"  Input:  {repr(text)}")
    print(f"  Output: {repr(normalized)}")
    print()

## 3. Language Filtering

`filter_english()` uses `langdetect` to classify each sentence and retain only
those detected as English (`lang == 'en'`).  Non-English rows are dropped and
the count is logged.  This prevents Comprehend from processing text it was
not trained on and reduces false positives.

In [ ]:
from langdetect import detect, LangDetectException

# Show language detection for each sentence.
for text in mock_sentences['text']:
    try:
        lang = detect(text)
    except LangDetectException:
        lang = 'unknown'
    print(f"  [{lang}] {text[:60]}")

In [ ]:
# Manually apply language filter (pandas simulation).
def is_english(text):
    try:
        return detect(text) == 'en'
    except LangDetectException:
        return False

english_mask = mock_sentences['text'].apply(is_english)
filtered_df = mock_sentences[english_mask].copy()
dropped = len(mock_sentences) - len(filtered_df)

print(f"Kept: {len(filtered_df)} rows")
print(f"Dropped: {dropped} rows")
filtered_df

## 4. Full Preprocessing Pipeline

The `preprocess()` function combines language filtering and normalization.
It adds a `text_normalized` column while preserving the original `text` column,
so that character offsets from Comprehend remain valid.

In [ ]:
# Simulate the pipeline on the filtered pandas DataFrame.
filtered_df['text_normalized'] = filtered_df['text'].apply(normalize_text)

print("Preprocessing result:")
filtered_df[['a_number', 'text', 'text_normalized']]

## Summary

- **Dropped**: 2 non-English rows (French, German)
- **Kept**: 3 English rows
- **Normalization**: `Müller` → `Muller` (diacritics removed, case preserved)
- The `text_normalized` column is what gets sent to Comprehend for NER inference.